In [21]:
from __future__ import annotations

import json
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


In [22]:
taskset_path = Path("../task/taskset35-updated2.csv")
raw_taskset_df = pd.read_csv(taskset_path)
taskset_df = raw_taskset_df.loc[raw_taskset_df["task_id"].notna()].copy()

print(f"Loaded {taskset_path.name}")
print(f"Raw CSV rows: {len(raw_taskset_df):,}")
print(f"Task rows after filtering to non-null task_id: {len(taskset_df):,}")
taskset_df.head()

Loaded taskset35-updated2.csv
Raw CSV rows: 35
Task rows after filtering to non-null task_id: 35


,task_id,prompt,note,ref_tool_call,answer_type,ground_truth,core_toolset,category,source_scope,temporal_req,complexity,ref_tool_call_steps,task_demand_profile
0,A1,how many posts does twitter collection have?,NaN,"{\r\n ""tool_name"": ""get_collection_summary""...",scalar,"{\r\n ""value"": 185873222,\r\n ""unit"": ""post"",\...",get_collection_summary,A,single,no,single,1,D
1,A2,Compare the harvesting date span of all availa...,NaN,"{\r\n ""tool_name"": ""get_collections"",\r\n ...",text,"{ ""summary"": ""GDELT has the longest harvesting...","get_collections, get_collection_summary",A,multi,no,multi,8,BR
2,A3,"Compare Reddit, Mastodon, and Twitter by total...",NaN,"{\r\n ""tool_name"": ""get_collection_summary""...",text,"{\r\n ""summary"": ""Twitter is the largest co...","get_collections, get_collection_summary",A,multi,no,multi,3,BR
3,A4,List all available collections and their start...,NaN,"[\r\n {\r\n ""tool_name"": ""get_collections""...",text,"{\r\n ""summary"": ""The available collections a...","get_collections, get_collection_summary",A,multi,no,multi,8,BR
4,A5,Which available collections have harvesting da...,NaN,"[\r\n {\r\n ""tool_name"": ""get_collections""...",text,"{\r\n ""summary"": ""Reddit, Bluesky, gdelt, Mas...","get_collections, get_collection_summary",A,multi,no,multi,8,BR


## Taskset overview

In [23]:
overview = pd.Series(
    {
        "raw_csv_rows": len(raw_taskset_df),
        "task_rows": len(taskset_df),
        "columns": taskset_df.shape[1],
        "unique_task_ids": taskset_df["task_id"].nunique(),
        "duplicate_task_ids": taskset_df["task_id"].duplicated().sum(),
        "total_missing_cells": int(taskset_df.isna().sum().sum()),
    },
    name="value",
)

display(overview.to_frame())
display(taskset_df.dtypes.rename("dtype").to_frame())

,value
raw_csv_rows,35
task_rows,35
columns,13
unique_task_ids,35
duplicate_task_ids,0
total_missing_cells,35


,dtype
task_id,str
prompt,str
note,float64
ref_tool_call,str
answer_type,str
ground_truth,str
core_toolset,str
category,str
source_scope,str
temporal_req,str


In [24]:
missingness = (
    taskset_df.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_pct=lambda x: x["missing_count"] / len(taskset_df))
    .sort_values(["missing_count", "missing_pct"], ascending=False)
)

missingness

,missing_count,missing_pct
note,35,1.0000
task_id,0,0.0000
prompt,0,0.0000
ref_tool_call,0,0.0000
answer_type,0,0.0000
ground_truth,0,0.0000
core_toolset,0,0.0000
category,0,0.0000
source_scope,0,0.0000
temporal_req,0,0.0000


In [25]:
distribution_df = taskset_df.assign(
    complexity_normalized=taskset_df["complexity"].str.lower(),
    source_scope_normalized=taskset_df["source_scope"].str.lower(),
)

distribution_cols = [
    "category",
    "source_scope_normalized",
    "temporal_req",
    "complexity_normalized",
    "answer_type",
    "ref_tool_call_steps",
    "task_demand_profile"
]

for col in distribution_cols:
    print(f"\n{col}")
    distribution = (
        distribution_df[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .to_frame("count")
        .assign(pct=lambda x: x["count"] / len(taskset_df))
    )
    display(distribution)



category


,count,pct
category,,
A,7,0.2000
B,7,0.2000
C,7,0.2000
D,7,0.2000
E,7,0.2000



source_scope_normalized


,count,pct
source_scope_normalized,,
single,23,0.6571
multi,12,0.3429



temporal_req


...

,count,pct
answer_type,,
text,22,0.6286
scalar,8,0.2286
chart,3,0.0857
timeseries,2,0.0571



ref_tool_call_steps


,count,pct
ref_tool_call_steps,,
1,15,0.4286
8,7,0.2000
2,7,0.2000
3,3,0.0857
5,2,0.0571
10,1,0.0286



task_demand_profile


,count,pct
task_demand_profile,,
S,13,0.3714
A,8,0.2286
D,7,0.2000
BR,7,0.2000


In [26]:
taskset_enriched = taskset_df.assign(
    complexity_normalized=taskset_df["complexity"].str.lower(),
    core_tool_count=taskset_df["core_toolset"].fillna("").str.split(",").map(
        lambda values: sum(bool(v.strip()) for v in values)
    ),
    prompt_length=taskset_df["prompt"].fillna("").str.len(),
)

numeric_summary = taskset_enriched[["ref_tool_call_steps", "core_tool_count", "prompt_length"]].describe()
display(numeric_summary)

display(
    taskset_enriched.groupby("complexity")[["ref_tool_call_steps", "core_tool_count", "prompt_length"]]
    .agg(["count", "mean", "median", "min", "max"])
    .sort_index()
)

,ref_tool_call_steps,core_tool_count,prompt_length
count,35.0000,35.0000,35.0000
mean,3.2571,1.4286,128.7714
std,2.9540,0.5021,44.7096
min,1.0000,1.0000,44.0000
25%,1.0000,1.0000,98.0000
50%,2.0000,1.0000,130.0000
75%,5.0000,2.0000,158.5000
max,10.0000,2.0000,215.0000


ref_tool_call_steps                       core_tool_count                       prompt_length                            
                         count   mean median min max           count   mean median min max         count     mean   median  min  max
complexity                                                                                                                          
Multi                        1 2.0000 2.0000   2   2               1 1.0000 1.0000   1   1             1 179.0000 179.0000  179  179
multi                       19 5.1053 5.0000   2  10              19 1.6842 2.0000   1   2            19 136.7368 132.0000   52  215
single                      15 1.0000 1.0000   1   1              15 1.1333 1.0000   1   2            15 115.3333 114.0000   44  213